# Gradient Boosting from Scratch

**Goal:** Implement gradient boosting for regression and classification from first principles in NumPy, then build a mini-XGBoost with regularization and gain-based splitting.

---

## Core Intuition

Gradient boosting is **sequential error correction**. Each new model (weak learner) is trained to predict the *residual errors* of the current ensemble.

- **Additive modeling:** $F_m(x) = F_{m-1}(x) + \eta \cdot h_m(x)$ where $h_m$ fits the negative gradient (pseudo-residuals)
- **Functional gradient descent:** We minimize loss in *function space*, not parameter space. Each weak learner takes a step in the direction that reduces the loss.
- For MSE loss, pseudo-residuals = actual residuals: $r_i^{(m)} = y_i - F_{m-1}(x_i)$
- For log-loss (classification), pseudo-residuals = $y_i - p_i$ where $p_i = \sigma(F_{m-1}(x_i))$

**Key formula (general):**
$$r_i^{(m)} = -\left[\frac{\partial L(y_i, F(x_i))}{\partial F(x_i)}\right]_{F=F_{m-1}}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

## 1. Decision Stump (Weak Learner)

A decision stump is a depth-1 decision tree: one split on one feature. This is the building block for our gradient boosting implementation.

In [ ]:
class DecisionStump:
    """Depth-1 decision tree for regression."""
    def __init__(self):
        self.feature_idx = None
        self.threshold = None
        self.left_value = None
        self.right_value = None
    
    def fit(self, X, y, sample_weight=None):
        n_samples, n_features = X.shape
        if sample_weight is None:
            sample_weight = np.ones(n_samples) / n_samples
        
        best_mse = np.inf
        
        for feat in range(n_features):
            thresholds = np.unique(X[:, feat])
            # Use midpoints between sorted unique values
            if len(thresholds) > 1:
                thresholds = (thresholds[:-1] + thresholds[1:]) / 2
            
            for thr in thresholds:
                left_mask = X[:, feat] <= thr
                right_mask = ~left_mask
                
                if left_mask.sum() == 0 or right_mask.sum() == 0:
                    continue
                
                # Weighted means
                w_left = sample_weight[left_mask]
                w_right = sample_weight[right_mask]
                left_val = np.average(y[left_mask], weights=w_left)
                right_val = np.average(y[right_mask], weights=w_right)
                
                # Weighted MSE
                pred = np.where(left_mask, left_val, right_val)
                mse = np.average((y - pred) ** 2, weights=sample_weight)
                
                if mse < best_mse:
                    best_mse = mse
                    self.feature_idx = feat
                    self.threshold = thr
                    self.left_value = left_val
                    self.right_value = right_val
        
        return self
    
    def predict(self, X):
        return np.where(
            X[:, self.feature_idx] <= self.threshold,
            self.left_value,
            self.right_value
        )

# Quick test
X_test = np.array([[1], [2], [3], [4], [5]], dtype=float)
y_test = np.array([1, 1, 0, 0, 0], dtype=float)
stump = DecisionStump().fit(X_test, y_test)
print(f"Split: feature {stump.feature_idx}, threshold {stump.threshold}")
print(f"Predictions: {stump.predict(X_test)}")

## 2. Gradient Boosting for Regression (MSE Loss)

For squared loss $L(y, F) = \frac{1}{2}(y - F)^2$:
- Pseudo-residuals: $r_i = y_i - F_{m-1}(x_i)$ (just the plain residuals)
- Each tree fits these residuals
- Shrinkage (learning rate $\eta$): $F_m(x) = F_{m-1}(x) + \eta \cdot h_m(x)$

In [ ]:
class GradientBoostingRegressor:
    """Gradient Boosting for regression with MSE loss."""
    
    def __init__(self, n_estimators=50, learning_rate=0.1, subsample=1.0):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.subsample = subsample  # fraction of samples for stochastic GB
        self.trees = []
        self.initial_prediction = None
        self.train_losses = []
    
    def fit(self, X, y):
        n_samples = X.shape[0]
        
        # Initialize with mean (minimizes MSE)
        self.initial_prediction = np.mean(y)
        F = np.full(n_samples, self.initial_prediction)
        
        self.stage_predictions = [F.copy()]  # for visualization
        self.train_losses = [np.mean((y - F) ** 2)]
        
        for m in range(self.n_estimators):
            # Compute pseudo-residuals (negative gradient of MSE)
            residuals = y - F
            
            # Subsampling for stochastic gradient boosting
            if self.subsample < 1.0:
                n_sub = max(1, int(n_samples * self.subsample))
                idx = np.random.choice(n_samples, n_sub, replace=False)
                X_sub, residuals_sub = X[idx], residuals[idx]
            else:
                X_sub, residuals_sub = X, residuals
            
            # Fit weak learner to pseudo-residuals
            tree = DecisionStump()
            tree.fit(X_sub, residuals_sub)
            self.trees.append(tree)
            
            # Update ensemble prediction
            F += self.learning_rate * tree.predict(X)
            
            self.stage_predictions.append(F.copy())
            self.train_losses.append(np.mean((y - F) ** 2))
        
        return self
    
    def predict(self, X):
        F = np.full(X.shape[0], self.initial_prediction)
        for tree in self.trees:
            F += self.learning_rate * tree.predict(X)
        return F

# Generate regression data
np.random.seed(42)
X_reg = np.sort(np.random.uniform(0, 10, 200)).reshape(-1, 1)
y_reg = np.sin(X_reg.ravel()) + 0.3 * np.random.randn(200)

# Fit
gb_reg = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1)
gb_reg.fit(X_reg, y_reg)

print(f"Final MSE: {gb_reg.train_losses[-1]:.4f}")

## 3. Visualize: How Each Tree Corrects Errors

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
stages = [0, 1, 5, 10, 30, 100]

for ax, stage in zip(axes.ravel(), stages):
    ax.scatter(X_reg, y_reg, alpha=0.3, s=10, color='gray', label='Data')
    ax.plot(X_reg, gb_reg.stage_predictions[stage], 'r-', linewidth=2, label='Prediction')
    ax.set_title(f'After {stage} trees (MSE={gb_reg.train_losses[stage]:.3f})')
    ax.set_ylim(-2, 2.5)
    ax.legend(fontsize=9)

plt.suptitle('Gradient Boosting: Sequential Error Correction', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Loss curve
plt.figure(figsize=(8, 4))
plt.plot(gb_reg.train_losses, 'b-', linewidth=2)
plt.xlabel('Boosting Round')
plt.ylabel('MSE')
plt.title('Training Loss Over Boosting Rounds')
plt.grid(True, alpha=0.3)
plt.show()

## 4. Gradient Boosting for Classification (Log Loss)

For binary classification with log-loss:
$$L(y, F) = -[y \log(p) + (1-y)\log(1-p)] \quad \text{where } p = \sigma(F)$$

Pseudo-residuals: $r_i = y_i - \sigma(F_{m-1}(x_i))$

Initialize $F_0 = \log\frac{\bar{y}}{1 - \bar{y}}$ (log-odds of the base rate).

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))

class GradientBoostingClassifier:
    """Gradient Boosting for binary classification with log loss."""
    
    def __init__(self, n_estimators=50, learning_rate=0.1, subsample=1.0):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.subsample = subsample
        self.trees = []
        self.initial_prediction = None
        self.train_losses = []
    
    def _log_loss(self, y, F):
        p = sigmoid(F)
        p = np.clip(p, 1e-15, 1 - 1e-15)
        return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
    
    def fit(self, X, y):
        n_samples = X.shape[0]
        
        # Initialize with log-odds
        p_mean = np.mean(y)
        self.initial_prediction = np.log(p_mean / (1 - p_mean + 1e-15))
        F = np.full(n_samples, self.initial_prediction)
        
        self.train_losses = [self._log_loss(y, F)]
        
        for m in range(self.n_estimators):
            # Pseudo-residuals for log loss
            p = sigmoid(F)
            residuals = y - p
            
            # Subsampling
            if self.subsample < 1.0:
                n_sub = max(1, int(n_samples * self.subsample))
                idx = np.random.choice(n_samples, n_sub, replace=False)
                X_sub, residuals_sub = X[idx], residuals[idx]
            else:
                X_sub, residuals_sub = X, residuals
            
            tree = DecisionStump()
            tree.fit(X_sub, residuals_sub)
            self.trees.append(tree)
            
            F += self.learning_rate * tree.predict(X)
            self.train_losses.append(self._log_loss(y, F))
        
        return self
    
    def predict_proba(self, X):
        F = np.full(X.shape[0], self.initial_prediction)
        for tree in self.trees:
            F += self.learning_rate * tree.predict(X)
        return sigmoid(F)
    
    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int)

# Demo on classification data
from sklearn.datasets import make_moons
X_cls, y_cls = make_moons(n_samples=300, noise=0.25, random_state=42)

gb_cls = GradientBoostingClassifier(n_estimators=100, learning_rate=0.3)
gb_cls.fit(X_cls, y_cls)

acc = np.mean(gb_cls.predict(X_cls) == y_cls)
print(f"Training accuracy: {acc:.4f}")
print(f"Final log-loss: {gb_cls.train_losses[-1]:.4f}")

In [ ]:
# Decision boundary visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Decision boundary
h = 0.05
x_min, x_max = X_cls[:, 0].min() - 0.5, X_cls[:, 0].max() + 0.5
y_min, y_max = X_cls[:, 1].min() - 0.5, X_cls[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
Z = gb_cls.predict_proba(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

axes[0].contourf(xx, yy, Z, levels=50, cmap='RdYlBu', alpha=0.8)
axes[0].scatter(X_cls[:, 0], X_cls[:, 1], c=y_cls, cmap='RdYlBu', edgecolors='k', s=20)
axes[0].set_title('GB Classifier Decision Boundary')

# Loss curve
axes[1].plot(gb_cls.train_losses, 'r-', linewidth=2)
axes[1].set_xlabel('Boosting Round')
axes[1].set_ylabel('Log Loss')
axes[1].set_title('Classification Log Loss Over Rounds')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Effect of Learning Rate (Shrinkage)

Lower learning rate = smaller steps = needs more trees but generalizes better. This is the **bias-variance tradeoff** in boosting.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

learning_rates = [1.0, 0.5, 0.1, 0.01]
colors = ['red', 'orange', 'blue', 'green']

for lr, color in zip(learning_rates, colors):
    gb = GradientBoostingRegressor(n_estimators=100, learning_rate=lr)
    gb.fit(X_reg, y_reg)
    axes[0].plot(gb.train_losses, color=color, label=f'lr={lr}', linewidth=2)
    axes[1].plot(X_reg, gb.predict(X_reg), color=color, label=f'lr={lr}', linewidth=1.5)

axes[0].set_xlabel('Boosting Round')
axes[0].set_ylabel('MSE')
axes[0].set_title('Effect of Learning Rate on Convergence')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].scatter(X_reg, y_reg, alpha=0.2, s=10, color='gray')
axes[1].set_title('Predictions with Different Learning Rates')
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Stochastic Gradient Boosting (Subsampling)

Using a random fraction of training data per boosting round reduces overfitting (similar to dropout intuition). Friedman (2002) showed `subsample=0.5` often works well.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for subsample, color in zip([1.0, 0.8, 0.5, 0.3], ['red', 'orange', 'blue', 'green']):
    gb = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, subsample=subsample)
    gb.fit(X_reg, y_reg)
    ax.plot(gb.train_losses, color=color, label=f'subsample={subsample}', linewidth=2)

ax.set_xlabel('Boosting Round')
ax.set_ylabel('MSE')
ax.set_title('Stochastic Gradient Boosting: Effect of Subsampling')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Mini-XGBoost: Regularized Gradient Boosting

XGBoost improves on vanilla gradient boosting with:

1. **Second-order approximation:** Uses both gradient $g_i$ and Hessian $h_i$ of the loss
2. **Regularization:** L2 penalty on leaf weights ($\lambda$), L1 penalty ($\alpha$), min gain to split ($\gamma$)
3. **Gain-based splitting:** 

$$\text{Gain} = \frac{1}{2}\left[\frac{G_L^2}{H_L + \lambda} + \frac{G_R^2}{H_R + \lambda} - \frac{(G_L+G_R)^2}{H_L+H_R+\lambda}\right] - \gamma$$

where $G = \sum g_i$, $H = \sum h_i$ over samples in the node.

**Optimal leaf weight:** $w^* = -\frac{G}{H + \lambda}$

In [ ]:
class XGBoostStump:
    """Decision stump using XGBoost-style gain-based splitting with regularization."""
    
    def __init__(self, lambda_reg=1.0, gamma=0.0):
        self.lambda_reg = lambda_reg  # L2 regularization on leaf weights
        self.gamma = gamma            # minimum gain to make a split
        self.feature_idx = None
        self.threshold = None
        self.left_weight = None
        self.right_weight = None
        self.no_split_weight = None
        self.did_split = False
    
    def _leaf_weight(self, g, h):
        """Optimal leaf weight: -G / (H + lambda)"""
        return -np.sum(g) / (np.sum(h) + self.lambda_reg)
    
    def _node_score(self, g, h):
        """Score of a node: G^2 / (H + lambda)"""
        return np.sum(g) ** 2 / (np.sum(h) + self.lambda_reg)
    
    def fit(self, X, g, h):
        """Fit using gradients g and hessians h (not labels directly)."""
        n_samples, n_features = X.shape
        
        self.no_split_weight = self._leaf_weight(g, h)
        parent_score = self._node_score(g, h)
        best_gain = -np.inf
        
        for feat in range(n_features):
            thresholds = np.unique(X[:, feat])
            if len(thresholds) > 1:
                thresholds = (thresholds[:-1] + thresholds[1:]) / 2
            
            for thr in thresholds:
                left_mask = X[:, feat] <= thr
                right_mask = ~left_mask
                
                if left_mask.sum() == 0 or right_mask.sum() == 0:
                    continue
                
                # XGBoost gain formula
                gain = 0.5 * (
                    self._node_score(g[left_mask], h[left_mask]) +
                    self._node_score(g[right_mask], h[right_mask]) -
                    parent_score
                ) - self.gamma
                
                if gain > best_gain:
                    best_gain = gain
                    self.feature_idx = feat
                    self.threshold = thr
                    self.left_weight = self._leaf_weight(g[left_mask], h[left_mask])
                    self.right_weight = self._leaf_weight(g[right_mask], h[right_mask])
        
        # Only split if gain is positive
        self.did_split = best_gain > 0
        return self
    
    def predict(self, X):
        if not self.did_split:
            return np.full(X.shape[0], self.no_split_weight)
        return np.where(
            X[:, self.feature_idx] <= self.threshold,
            self.left_weight,
            self.right_weight
        )


class MiniXGBoost:
    """Mini XGBoost implementation for regression (MSE loss with second-order)."""
    
    def __init__(self, n_estimators=100, learning_rate=0.1, lambda_reg=1.0, gamma=0.0, subsample=1.0):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.lambda_reg = lambda_reg
        self.gamma = gamma
        self.subsample = subsample
        self.trees = []
        self.initial_prediction = None
        self.train_losses = []
    
    def fit(self, X, y):
        n_samples = X.shape[0]
        self.initial_prediction = np.mean(y)
        F = np.full(n_samples, self.initial_prediction)
        self.train_losses = [np.mean((y - F) ** 2)]
        
        for m in range(self.n_estimators):
            # For MSE: gradient = -(y - F) = F - y, hessian = 1
            # (XGBoost convention: minimize loss, so gradient is dL/dF)
            g = F - y           # first-order gradient
            h = np.ones_like(y)  # second-order hessian (constant for MSE)
            
            # Subsampling
            if self.subsample < 1.0:
                n_sub = max(1, int(n_samples * self.subsample))
                idx = np.random.choice(n_samples, n_sub, replace=False)
            else:
                idx = np.arange(n_samples)
            
            tree = XGBoostStump(lambda_reg=self.lambda_reg, gamma=self.gamma)
            tree.fit(X[idx], g[idx], h[idx])
            self.trees.append(tree)
            
            F += self.learning_rate * tree.predict(X)
            self.train_losses.append(np.mean((y - F) ** 2))
        
        return self
    
    def predict(self, X):
        F = np.full(X.shape[0], self.initial_prediction)
        for tree in self.trees:
            F += self.learning_rate * tree.predict(X)
        return F

# Compare vanilla GB vs mini-XGBoost
xgb = MiniXGBoost(n_estimators=100, learning_rate=0.1, lambda_reg=1.0, gamma=0.0)
xgb.fit(X_reg, y_reg)

print(f"Vanilla GB final MSE: {gb_reg.train_losses[-1]:.4f}")
print(f"Mini-XGBoost final MSE: {xgb.train_losses[-1]:.4f}")

In [ ]:
# Visualize regularization effect
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Different lambda values
for lam, color in zip([0.0, 1.0, 5.0, 20.0], ['red', 'orange', 'blue', 'green']):
    xgb_temp = MiniXGBoost(n_estimators=100, learning_rate=0.1, lambda_reg=lam)
    xgb_temp.fit(X_reg, y_reg)
    axes[0].plot(xgb_temp.train_losses, color=color, label=f'lambda={lam}', linewidth=2)

axes[0].set_xlabel('Boosting Round')
axes[0].set_ylabel('MSE')
axes[0].set_title('Effect of L2 Regularization (lambda)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Different gamma values
for gam, color in zip([0.0, 0.1, 0.5, 2.0], ['red', 'orange', 'blue', 'green']):
    xgb_temp = MiniXGBoost(n_estimators=100, learning_rate=0.1, gamma=gam)
    xgb_temp.fit(X_reg, y_reg)
    axes[1].plot(xgb_temp.train_losses, color=color, label=f'gamma={gam}', linewidth=2)

axes[1].set_xlabel('Boosting Round')
axes[1].set_ylabel('MSE')
axes[1].set_title('Effect of Min Split Gain (gamma)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Compare with sklearn

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor as SklearnGB
from sklearn.metrics import mean_squared_error

# sklearn
sk_gb = SklearnGB(n_estimators=100, learning_rate=0.1, max_depth=1, random_state=42)
sk_gb.fit(X_reg, y_reg)
sk_pred = sk_gb.predict(X_reg)

# Ours
our_pred = gb_reg.predict(X_reg)

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(X_reg, y_reg, alpha=0.3, s=10, color='gray', label='Data')
ax.plot(X_reg, our_pred, 'b-', linewidth=2, label=f'Ours (MSE={mean_squared_error(y_reg, our_pred):.4f})')
ax.plot(X_reg, sk_pred, 'r--', linewidth=2, label=f'sklearn (MSE={mean_squared_error(y_reg, sk_pred):.4f})')
ax.legend()
ax.set_title('Our Gradient Boosting vs sklearn')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Our MSE:     {mean_squared_error(y_reg, our_pred):.4f}")
print(f"sklearn MSE: {mean_squared_error(y_reg, sk_pred):.4f}")

## 9. Hyperparameter Tuning Strategy (Practical Notes)

Typical tuning order for gradient boosting (XGBoost/LightGBM):

1. **Start with:** `learning_rate=0.1`, `n_estimators=100-1000` (use early stopping)
2. **Tune tree structure:** `max_depth` (3-8), `min_child_weight` (controls minimum H in leaf)
3. **Tune regularization:** `lambda` (L2), `alpha` (L1), `gamma` (min split gain)
4. **Tune stochasticity:** `subsample` (0.6-0.9), `colsample_bytree` (0.6-0.9)
5. **Final:** Lower `learning_rate`, increase `n_estimators` proportionally

**Rule of thumb:** `learning_rate * n_estimators` should stay roughly constant.

In [ ]:
# Demonstrate the learning_rate vs n_estimators tradeoff
fig, ax = plt.subplots(figsize=(10, 5))

configs = [
    (0.5, 20, 'red'),
    (0.1, 100, 'blue'),
    (0.05, 200, 'green'),
    (0.01, 500, 'purple'),
]

ax.scatter(X_reg, y_reg, alpha=0.2, s=10, color='gray')
for lr, n_est, color in configs:
    gb = GradientBoostingRegressor(n_estimators=n_est, learning_rate=lr)
    gb.fit(X_reg, y_reg)
    pred = gb.predict(X_reg)
    mse = mean_squared_error(y_reg, pred)
    ax.plot(X_reg, pred, color=color, linewidth=2,
            label=f'lr={lr}, n={n_est} (MSE={mse:.3f})')

ax.set_title('Learning Rate vs Number of Estimators (product ~ 10)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## Interview Questions

### "Boosting vs Bagging?"
- **Bagging** (e.g., Random Forest): Trains models **independently in parallel**, combines by averaging. Reduces **variance**.
- **Boosting** (e.g., GBDT, XGBoost): Trains models **sequentially**, each correcting errors of the previous. Reduces **bias** (and can reduce variance too with regularization).
- Bagging rarely overfits with more trees; boosting can overfit if not regularized.

### "Why does gradient boosting overfit more?"
- Sequential nature means it keeps focusing on hard examples (noise).
- With too many rounds and high learning rate, it memorizes training noise.
- Mitigation: shrinkage (low learning rate), early stopping, subsampling, regularization.

### "XGBoost vs vanilla Gradient Boosting --- what's different?"
1. **Second-order Taylor expansion** of the loss (uses Hessian, not just gradient)
2. **Regularization** built into objective: L1/L2 on leaf weights, gamma for min gain
3. **System optimizations:** column block structure, cache-aware access, parallel split finding
4. **Sparsity awareness:** handles missing values natively
5. **Approximate split finding** with quantile sketch for large data

### "Second-order approximation in XGBoost"
The objective is approximated with a second-order Taylor expansion:
$$\text{obj}^{(t)} \approx \sum_{i=1}^{n} [g_i f_t(x_i) + \frac{1}{2} h_i f_t(x_i)^2] + \Omega(f_t)$$
where $g_i = \partial_{\hat{y}^{(t-1)}} L(y_i, \hat{y}^{(t-1)})$ and $h_i = \partial^2_{\hat{y}^{(t-1)}} L(y_i, \hat{y}^{(t-1)})$.
This gives a closed-form optimal leaf weight and a principled gain formula that works for *any* differentiable loss.

### Connection to Functional Gradient Descent
Gradient boosting performs gradient descent in **function space**. The ensemble $F_m$ is updated by adding a function $h_m$ that points in the direction of steepest descent of the expected loss. The learning rate controls the step size. This is why Friedman originally called it "Gradient Boosting Machine" -- it's a gradient descent procedure where the "parameters" are functions.

In [ ]:
print("Notebook 08 complete: Gradient Boosting from Scratch")
print("="*55)
print("Implemented:")
print("  - Decision stump weak learner")
print("  - Gradient Boosting for regression (MSE)")
print("  - Gradient Boosting for classification (log loss)")
print("  - Shrinkage / learning rate")
print("  - Stochastic gradient boosting (subsampling)")
print("  - Mini-XGBoost with L2 reg + gain-based splitting")
print("  - Validated against sklearn")